In [1]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from vllm import LLM, SamplingParams

# VLLM 모델 및 토크나이저 로드
BASE_MODEL = "cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0"
llm = LLM(model=BASE_MODEL, max_model_len=4096,tensor_parallel_size=8,gpu_memory_utilization=0.2)

# instruction 컬럼을 순차적으로 가져와서 모델로 답변 생성 후 rejected 컬럼에 추가
output = []

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-07-29 10:17:00,775	INFO worker.py:1770 -- Started a local Ray instance.


INFO 07-29 10:17:01 config.py:623] Defaulting to use mp for distributed inference
INFO 07-29 10:17:01 llm_engine.py:161] Initializing an LLM engine (v0.5.0.post1) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
(VllmWorkerProcess pid=3673852) INFO 07-29 10:17:05 multiproc_worker_utils.py:215] Worker ready; awaiting tasks
(VllmWorkerProcess pid=3673857

Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/psm_1c799a2a'
Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/psm_1c799a2a'
Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/psm_1c799a2a'
Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
KeyError: '/psm_1c799a2a'
Traceback (most recent call last):
  File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/multiprocessing/resource_tracker.py", line 209, in main
    cache[rtype].remove(name)
K

(VllmWorkerProcess pid=3673852) (VllmWorkerProcess pid=3673857) (VllmWorkerProcess pid=3673858) INFO 07-29 10:17:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/ubuntu/.config/vllm/gpu_p2p_access_cache_for_0,1,2,3,4,5,6,7.json
(VllmWorkerProcess pid=3673854) INFO 07-29 10:17:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/ubuntu/.config/vllm/gpu_p2p_access_cache_for_0,1,2,3,4,5,6,7.json
INFO 07-29 10:17:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/ubuntu/.config/vllm/gpu_p2p_access_cache_for_0,1,2,3,4,5,6,7.json
INFO 07-29 10:17:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/ubuntu/.config/vllm/gpu_p2p_access_cache_for_0,1,2,3,4,5,6,7.json
INFO 07-29 10:17:10 weight_utils.py:218] Using model weights format ['*.safetensors']
(VllmWorkerProcess pid=3673855) INFO 07-29 10:17:10 weight_utils.py:218] Using model weights format ['*.safetensors']
(VllmWorkerProcess pid=3673857) INFO

In [61]:
# CSV 파일 불러오기
import pandas as pd
df = pd.read_csv('./data/추론2000_final.csv')
df

,instruction,option
0,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,NaN
1,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,NaN
2,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,NaN
3,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,NaN
4,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,NaN
...,...,...
1995,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,NaN
1996,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,NaN
1997,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,NaN
1998,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,NaN


In [62]:
system_prompt = """
당신의 목표는 일상대화[Conversation]와 문제[Question], 답변[Answer]을 보고 옵션값을 생성하는것입니다. 답변은 한국어로 출력합니다.

출력은 다음 지침을 준수하십시오:
1. 먼저 두 화자가 무슨 주제를 가지고 대화 중인지 파악한 후에
2. 주제와 관련된 3가지 옵션을 출력해주세요. 옵션에는 꼭 [Answer]이  포함되어야 하며 A, B, C 옵션으로 출력합니다.
3. 3가지 옵션 중 정답은 랜덤으로 들어가며 정답은 항상 생성된 옵션 중 하나입니다.
4. 각 옵션은 명확히 구분되고, 가능한 한 다양한 방식으로 표현되도록 합니다.

예시입니다.

[Conversation]
화자2: 흠ㅋㅋㅋ 어쨌든 이제 카페 자주 들어가서 ... 쫓겨나지 않게 잘 버텨봐야죠
화자1: ㅋㅋㅋㅋㅋㅋㅋ 아 맞다 저 인증하는거 까먹고 있었는데
화자1: 지금 해야겠네요..
화자2: 그거 까딱하면 까먹어버려요 빨리 빨리 해 버려야 해요 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
화자1: 맞아요.. 입금으로 대충 때우고
화자2: 카페는 보통 어떻게 활용하고 계신가요? 주변에 거기 가입한 다른 분이 없어서 궁금해요
화자1: 저는 그냥 정보만 확인하는 정도에요! 아니면 제가 정보글을 올리거나요
화자2: 앗 저도 거의 확인용... 좋은 정보 공유하는 멋진 회원이 되고 싶은데
화자2: 현실은 줍줍 인간이네요
화자1: ㅋㅋㅋㅋㅋㅋ 근데 올릴 게 딱히 없어요..
화자1: 주식하는 사람도 많다던데 전 주식을 안 해서
화자2: 그럼 어떤 정보 주로 확인하시나요??
화자1: 저 그냥 이것저것 올라오는 거요 ㅋㅋㅋㅋ 이런 앱테크도 확인하고..
화자2: 아하ㅋㅋㅋㅋㅋㅋ 저는 주식 공부 시작한 지 이제 딱 일 년 좀 넘어가요
화자1: 주식 공부 저는 미루는 중이에요 현생이 너무 바빠서 ㅜㅜ
화자2: 아하 현생... 공부할 것이 많으시지요 참ㅠㅠㅠ
화자2: 저도 얄팍하게만 공부해두고 망할 것 같지 않은 주식에만... 적금처럼 넣고 있어요
화자1: 망할 것 같지 않은ㅋㅋㅋㅋ

[Question] 다음 중 (원인)에 해당하는 것은 무엇인가요?

[Answer] 화자2는 주식을 잘 알지 못한다.

[Option]
A. 화자1은 조심스럽게 주식 투자를 하는 화자2가 귀엽다고 생각한다.
B. 화자2는 주식을 잘 알지 못한다.
C. 화자2는 주식 공부를 시작한 지 이제 딱 일 년 좀 넘었다.

[정답] 
B

위와 같이 Option을 생성해주세요
"""

In [67]:
# 샘플링 파라미터 설정
sampling_params = SamplingParams(temperature=0, max_tokens=1024)

for instruction in tqdm(df['instruction']):
    
    # 템플릿에 instruction 삽입
    text = system_prompt + instruction
    
    # 답변 생성
    outputs = llm.generate([text], sampling_params)
    
    # 생성된 텍스트 추출
    response = outputs[0].outputs[0].text.strip()
    
    # rejected_responses 리스트에 답변 추가
    output.append(response)

# 새로운 rejected 컬럼 추가
df['option'] = output

100%|██████████| 2000/2000 [21:04<00:00,  1.58it/s]


(VllmWorkerProcess pid=3673852) ERROR 07-29 12:03:23 multiproc_worker_utils.py:226] Exception in worker VllmWorkerProcess while processing method start_worker_execution_loop: [../third_party/gloo/gloo/transport/tcp/unbound_buffer.cc:81] Timed out waiting 1800000ms for recv operation to complete, Traceback (most recent call last):
(VllmWorkerProcess pid=3673852) ERROR 07-29 12:03:23 multiproc_worker_utils.py:226]   File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/vllm/executor/multiproc_worker_utils.py", line 223, in _run_worker_process
(VllmWorkerProcess pid=3673852) ERROR 07-29 12:03:23 multiproc_worker_utils.py:226]     output = executor(*args, **kwargs)
(VllmWorkerProcess pid=3673852) ERROR 07-29 12:03:23 multiproc_worker_utils.py:226]   File "/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
(VllmWorkerProcess pid=3673852) ERROR 07-29 12:03:23 multiproc_worker_utils.py:226]     return func

In [68]:
df

,instruction,option
0,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,[Option]\nA. 국민 지원금 신청 자격은 아주 까다로운 편이다.\nB. 국민...
1,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,[Option]\nA. 화자1은 국민 지원금 신청 자격 기준에 대해 토론할 것이다....
2,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,[Option]\nA. 화자1은 국민지원금을 신청할 수 있다.\nB. 화자2는 자녀...
3,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,[Option]\nA. 화자1은 국민 지원금의 사용처에 대한 불만을 표현한다.\nB...
4,[Conversation]\n화자1: 국민지원금 신청 일이 다가오네요\n화자2: 아...,[Option]\nA. 화자1은 국민 지원금 신청 자격이 지나치게 까다로운 것이 황...
...,...,...
1995,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,[Option]\nA. 화자1은 룸메이트와 잘 지냈다고 언급했다.\nB. 화자1은 ...
1996,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,[Option]\nA. 화자2는 룸메이트와의 갈등을 해결하기 위해 노력했다.\nB....
1997,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,[Option]\nA. 화자1은 룸메이트와 싸운 적이 있다.\nB. 화자1은 룸메이...
1998,[Conversation]\n화자1: 기숙사 생활하신 경험이 있으신가요?\n화자2:...,[Option]\nA. 화자1은 단체 생활 경험이 나쁘지 않았다.\nB. 화자2는 ...


In [69]:
df.to_csv('test.csv',index=False)

In [70]:
import pandas as pd
df = pd.read_csv('test.csv')
df['option']

0       [Option]\nA. 국민 지원금 신청 자격은 아주 까다로운 편이다.\nB. 국민...
1       [Option]\nA. 화자1은 국민 지원금 신청 자격 기준에 대해 토론할 것이다....
2       [Option]\nA. 화자1은 국민지원금을 신청할 수 있다.\nB. 화자2는 자녀...
3       [Option]\nA. 화자1은 국민 지원금의 사용처에 대한 불만을 표현한다.\nB...
4       [Option]\nA. 화자1은 국민 지원금 신청 자격이 지나치게 까다로운 것이 황...
                              ...                        
1995    [Option]\nA. 화자1은 룸메이트와 잘 지냈다고 언급했다.\nB. 화자1은 ...
1996    [Option]\nA. 화자2는 룸메이트와의 갈등을 해결하기 위해 노력했다.\nB....
1997    [Option]\nA. 화자1은 룸메이트와 싸운 적이 있다.\nB. 화자1은 룸메이...
1998    [Option]\nA. 화자1은 단체 생활 경험이 나쁘지 않았다.\nB. 화자2는 ...
1999    [Option]\nA. 화자1은 룸메이트와의 갈등을 겪은 화자2를 안타깝게 생각한다...
Name: option, Length: 2000, dtype: object

In [26]:
json = pd.read_json('./일상대화-Data/일상대화요약_test.json')
json['output'] = df['output']
json

FileNotFoundError: File ./일상대화-Data/일상대화요약_test.json does not exist

In [ ]:
# DataFrame을 JSON 파일로 저장, orient='records'와 lines=True 옵션 사용
json.to_json('일상대화_full_template.json', force_ascii=False, orient='records', lines=True)